In [31]:
import pandas as pd
import altair as alt

1. Crear dataframe con los datos de SIDER 4.1

In [32]:
df = pd.read_csv('meddra_all_se.tsv', sep = '\t', usecols = [1, 4, 5], header = None) # Crear df importando datos del archivo (sólo cols de interés), separándolos por una sangría y sin headers por columna
display(df) # Print con mejor visualización

,1,4,5
0,CID000010917,C0000729,Abdominal cramps
1,CID000010917,C0000737,Abdominal pain
2,CID000010917,C0000737,Abdominal pain
3,CID000010917,C0687713,Gastrointestinal pain
4,CID000010917,C0000737,Abdominal pain
...,...,...,...
309844,CID071306834,C1145670,Respiratory failure
309845,CID071306834,C3665386,Abnormal vision
309846,CID071306834,C3665347,Visual impairment
309847,CID071306834,C3665596,Warts


*ID fármaco: STITCH / ID RAF: MedDra*

In [33]:
df.rename(columns = {1:'ID_fármaco', 4:'ID_RAF', 5:'Nombre_RAF'}, inplace = True) # Cambiar nombre a las columnas
display(df) # Print con mejor visualización

,ID_fármaco,ID_RAF,Nombre_RAF
0,CID000010917,C0000729,Abdominal cramps
1,CID000010917,C0000737,Abdominal pain
2,CID000010917,C0000737,Abdominal pain
3,CID000010917,C0687713,Gastrointestinal pain
4,CID000010917,C0000737,Abdominal pain
...,...,...,...
309844,CID071306834,C1145670,Respiratory failure
309845,CID071306834,C3665386,Abnormal vision
309846,CID071306834,C3665347,Visual impairment
309847,CID071306834,C3665596,Warts


2. Modificar el ID del fármaco de acuerdo a bases de datos estandarizadas

In [34]:
def cambiarID(cid): # Función que permite eliminar todos los 'CID' de la columna 'ID fármaco'; devuelve sólo el número. Modifica el ID de STITCH al de PubChem
    assert cid.startswith('CID') # Confirmar que inicia con CID
    return int(cid[3:]) # Devolver todos los elementos después de las primeras 3 letras (CID)

*ID fármaco: PubChem / ID RAF: MedDra*

In [35]:
url = 'https://raw.githubusercontent.com/dhimmel/drugbank/3e87872db5fca5ac427ce27464ab945c0ceb4ec6/data/mapping/pubchem.tsv' # Importar datos de un URL
df1 = pd.read_csv(url, sep = '\t') # Crear df importando datos del archivo, separándolos por una sangría
display(df1) # Print con mejor visualización

,drugbank_id,pubchem_id
0,DB00014,11980055
1,DB00014,11981235
2,DB00014,11982741
3,DB00014,16052011
4,DB00014,23581804
...,...,...
204704,DB09028,74070157
204705,DB09028,74834862
204706,DB09028,77513518
204707,DB09028,87355970


*ID fármaco: DrugBank // PubChem / ID RAF: MedDra*

In [36]:
df1.rename(columns = {'drugbank_id':'ID_med', 'pubchem_id':'ID_fármaco'}, inplace = True) # Cambiar nombre a las columnas (para poder juntar estas columnas posteriormente)
display(df1) # Print con mejor visualización

,ID_med,ID_fármaco
0,DB00014,11980055
1,DB00014,11981235
2,DB00014,11982741
3,DB00014,16052011
4,DB00014,23581804
...,...,...
204704,DB09028,74070157
204705,DB09028,74834862
204706,DB09028,77513518
204707,DB09028,87355970


In [37]:
df['ID_fármaco'] = df.ID_fármaco.map(cambiarID) # Utiliza la función con los datos de la columna 'ID_fármaco' y sustituye los datos en dicha columna
df = df1.merge(df) # Une los datos de las columnas con la misma información, en este caso mapea los números de ID_fármaco con su correspondencia a ID_fármaco_2
df = df.drop(['ID_fármaco'], axis=1)
display(df) # Print con mejor visualización

,ID_med,ID_RAF,Nombre_RAF
0,DB00014,C0000737,Abdominal pain
1,DB00014,C0687713,Gastrointestinal pain
2,DB00014,C0000737,Abdominal pain
3,DB00014,C0002170,Alopecia
4,DB00014,C0002170,Alopecia
...,...,...,...
301158,DB09020,C0159066,Abdominal rigidity
301159,DB09020,C1321898,Blood in stool
301160,DB09020,C0018932,Haematochezia
301161,DB09020,C2242737,Anorectal discomfort


In [38]:
df = df.dropna() # Eliminar filas sin información
df = df.drop_duplicates(['ID_med', 'ID_RAF']) # Elimina información duplicada en las columnas indicadas
display(df) # Print con mejor visualización

,ID_med,ID_RAF,Nombre_RAF
0,DB00014,C0000737,Abdominal pain
1,DB00014,C0687713,Gastrointestinal pain
3,DB00014,C0002170,Alopecia
5,DB00014,C0002418,Amblyopia
7,DB00014,C0002453,Amenorrhoea
...,...,...,...
301155,DB09020,C0232487,Abdominal discomfort
301157,DB09020,C0948733,Abdominal spasm
301158,DB09020,C0159066,Abdominal rigidity
301159,DB09020,C1321898,Blood in stool


3. Añadir el nombre del fármaco de acuerdo a su ID (DrugBank)

In [39]:
url = 'https://raw.githubusercontent.com/dhimmel/drugbank/3e87872db5fca5ac427ce27464ab945c0ceb4ec6/data/drugbank.tsv' # Importar datos de un URL
df2 = pd.read_csv(url, sep = '\t', usecols = [0, 1]) # Crear df importando datos del archivo (sólo cols de interés), separándolos por una sangría
display(df2) # Print con mejor visualización

,drugbank_id,name
0,DB00001,Lepirudin
1,DB00002,Cetuximab
2,DB00003,Dornase alfa
3,DB00004,Denileukin diftitox
4,DB00005,Etanercept
...,...,...
7754,DB09023,Benactyzine
7755,DB09024,Follitropin Alpha
7756,DB09026,Aliskiren
7757,DB09028,Cytisine


In [40]:
df2.rename(columns = {'drugbank_id':'ID_med', 'name':'Nombre_med'}, inplace = True) # Cambiar nombre a las columnas (para poder juntar estas columnas posteriormente)
display(df2) # Print con mejor visualización

,ID_med,Nombre_med
0,DB00001,Lepirudin
1,DB00002,Cetuximab
2,DB00003,Dornase alfa
3,DB00004,Denileukin diftitox
4,DB00005,Etanercept
...,...,...
7754,DB09023,Benactyzine
7755,DB09024,Follitropin Alpha
7756,DB09026,Aliskiren
7757,DB09028,Cytisine


In [41]:
df = df2.merge(df) # Une los datos de las columnas con la misma información, en este caso mapea los números de ID_med con su correspondencia a Nombre_med
display(df) # Print con mejor visualización

,ID_med,Nombre_med,ID_RAF,Nombre_RAF
0,DB00014,Goserelin,C0000737,Abdominal pain
1,DB00014,Goserelin,C0687713,Gastrointestinal pain
2,DB00014,Goserelin,C0002170,Alopecia
3,DB00014,Goserelin,C0002418,Amblyopia
4,DB00014,Goserelin,C0002453,Amenorrhoea
...,...,...,...,...
153658,DB09020,Bisacodyl,C0232487,Abdominal discomfort
153659,DB09020,Bisacodyl,C0948733,Abdominal spasm
153660,DB09020,Bisacodyl,C0159066,Abdominal rigidity
153661,DB09020,Bisacodyl,C1321898,Blood in stool


4. Contar efectos adversos por fármaco

In [42]:
final = df.value_counts("Nombre_med", sort = True) # Cuenta el número de filas asociadas a cada valor único de la columna indicada y los ordena en orden descendente
print(final) # Imprime el resultado

Nombre_med
Pregabalin                               839
Aripiprazole                             827
Escitalopram                             823
Citalopram                               823
Ropinirole                               682
                                        ... 
Mepyramine                                 1
Oxtriphylline                              1
Choline                                    1
5-Methyl-5,6,7,8-Tetrahydrofolic Acid      1
5-methyltetrahydrofolate                   1
Length: 1223, dtype: int64


In [43]:
df_final = final.rename_axis('Nombre_med').reset_index(name='#_RAFs') # Convierte la estructura de datos 'final' en un dataframe
display(df_final) # Print con mejor visualización

,Nombre_med,#_RAFs
0,Pregabalin,839
1,Aripiprazole,827
2,Escitalopram,823
3,Citalopram,823
4,Ropinirole,682
...,...,...
1218,Mepyramine,1
1219,Oxtriphylline,1
1220,Choline,1
1221,"5-Methyl-5,6,7,8-Tetrahydrofolic Acid",1


In [44]:
df_final["Nombre_med"] = df_final["Nombre_med"].str.lower() # Convierte todas las palabras en la columna indicada en minúsculas
df_final["Nombre_med"] = df_final["Nombre_med"].str.capitalize() # Convierte todas las palabras en la columna indicada con la 1ra letra mayúscula

5. Graficar efectos adversos por fármaco

In [45]:
alt.Chart(df_final, title = alt.TitleParams('Número de efectos adversos por medicamento en SIDER 4.1', anchor='middle')).mark_bar().encode(x = alt.X('Nombre_med', title = "Nombre del medicamento"), y = alt.Y('#_RAFs', title = "Número de reacciones adversas por medicamento"), color = alt.value('blue'))

alt.Chart(...)

6. Separar fármacos por categorías

In [46]:
url = 'https://raw.githubusercontent.com/fabkury/atcd/master/WHO%20ATC-DDD%202021-12-03.csv' # Importar datos de un URL
df3 = pd.read_csv(url, sep = ',', usecols = [0,1]) # Crear df importando datos del archivo (sólo cols de interés), separándolos por una coma
display(df3) # Print con mejor visualización

,atc_code,atc_name
0,A,ALIMENTARY TRACT AND METABOLISM
1,A01,STOMATOLOGICAL PREPARATIONS
2,A01A,STOMATOLOGICAL PREPARATIONS
3,A01AA,Caries prophylactic agents
4,A01AA01,sodium fluoride
...,...,...
6966,V10XX01,sodium phosphate (32P)
6967,V10XX02,ibritumomab tiuxetan (90Y)
6968,V10XX03,radium (223Ra) dichloride
6969,V10XX04,lutetium (177Lu) oxodotreotide


In [47]:
df3.rename(columns = {'atc_code':'ID_ATC', 'atc_name':'Nombre_med'}, inplace = True) # Cambiar nombre a las columnas (para poder juntar estas columnas posteriormente)
display(df3) # Print con mejor visualización

,ID_ATC,Nombre_med
0,A,ALIMENTARY TRACT AND METABOLISM
1,A01,STOMATOLOGICAL PREPARATIONS
2,A01A,STOMATOLOGICAL PREPARATIONS
3,A01AA,Caries prophylactic agents
4,A01AA01,sodium fluoride
...,...,...
6966,V10XX01,sodium phosphate (32P)
6967,V10XX02,ibritumomab tiuxetan (90Y)
6968,V10XX03,radium (223Ra) dichloride
6969,V10XX04,lutetium (177Lu) oxodotreotide


In [48]:
df3["Nombre_med"] = df3["Nombre_med"].str.lower()
df3["Nombre_med"] = df3["Nombre_med"].str.capitalize()
display(df3)

,ID_ATC,Nombre_med
0,A,Alimentary tract and metabolism
1,A01,Stomatological preparations
2,A01A,Stomatological preparations
3,A01AA,Caries prophylactic agents
4,A01AA01,Sodium fluoride
...,...,...
6966,V10XX01,Sodium phosphate (32p)
6967,V10XX02,Ibritumomab tiuxetan (90y)
6968,V10XX03,Radium (223ra) dichloride
6969,V10XX04,Lutetium (177lu) oxodotreotide


In [49]:
df_final = df3.merge(df_final) # Une los datos de las columnas con la misma información, en este caso mapea los números de ID_med con su correspondencia a Nombre_med
display(df_final) # Print con mejor visualización

,ID_ATC,Nombre_med,#_RAFs
0,A01AB03,Chlorhexidine,59
1,B05CA02,Chlorhexidine,59
2,D08AC02,Chlorhexidine,59
3,D09AA12,Chlorhexidine,59
4,R02AA05,Chlorhexidine,59
...,...,...,...
1566,V08AB09,Iodixanol,151
1567,V08CA03,Gadodiamide,109
1568,V08CA04,Gadoteridol,76
1569,V08CA06,Gadoversetamide,203


In [50]:
def fsubclase(id): # Función que permite encontrar la clase de la columna 'ID_ATC'; devuelve sólo el número de dicha clase según el ATC, elimina la particularidad del fármaco. 
    return id[:5] # Devolver los primeros 5 elementos

In [51]:
df_final['ID_ATC_subclase'] = df_final.ID_ATC.map(fsubclase) # Utiliza la función con los datos de la columna 'ID_ATC' y crea una nueva columna con los datos obtenidos
display(df_final) # Print con mejor visualización

,ID_ATC,Nombre_med,#_RAFs,ID_ATC_subclase
0,A01AB03,Chlorhexidine,59,A01AB
1,B05CA02,Chlorhexidine,59,B05CA
2,D08AC02,Chlorhexidine,59,D08AC
3,D09AA12,Chlorhexidine,59,D09AA
4,R02AA05,Chlorhexidine,59,R02AA
...,...,...,...,...
1566,V08AB09,Iodixanol,151,V08AB
1567,V08CA03,Gadodiamide,109,V08CA
1568,V08CA04,Gadoteridol,76,V08CA
1569,V08CA06,Gadoversetamide,203,V08CA


In [52]:
col0 = df_final.pop('ID_ATC_subclase') # Crear objeto para reubicar la columna 'ID_ATC_subclase'
df_final.insert(0, 'ID_ATC_subclase', col0) # Reubicar la columna 'ID_ATC_subclase'
display(df_final) # Print con mejor visualización

,ID_ATC_subclase,ID_ATC,Nombre_med,#_RAFs
0,A01AB,A01AB03,Chlorhexidine,59
1,B05CA,B05CA02,Chlorhexidine,59
2,D08AC,D08AC02,Chlorhexidine,59
3,D09AA,D09AA12,Chlorhexidine,59
4,R02AA,R02AA05,Chlorhexidine,59
...,...,...,...,...
1566,V08AB,V08AB09,Iodixanol,151
1567,V08CA,V08CA03,Gadodiamide,109
1568,V08CA,V08CA04,Gadoteridol,76
1569,V08CA,V08CA06,Gadoversetamide,203


In [53]:
df3.rename(columns = {'ID_ATC':'ID_ATC_subclase', 'Nombre_med':'Nombre_ATC_subclase'}, inplace = True) # Cambiar nombre a las columnas (para poder juntar estas columnas posteriormente)
display(df3) # Print con mejor visualización

,ID_ATC_subclase,Nombre_ATC_subclase
0,A,Alimentary tract and metabolism
1,A01,Stomatological preparations
2,A01A,Stomatological preparations
3,A01AA,Caries prophylactic agents
4,A01AA01,Sodium fluoride
...,...,...
6966,V10XX01,Sodium phosphate (32p)
6967,V10XX02,Ibritumomab tiuxetan (90y)
6968,V10XX03,Radium (223ra) dichloride
6969,V10XX04,Lutetium (177lu) oxodotreotide


In [54]:
df_final = df3.merge(df_final) # Une los datos de las columnas con la misma información, en este caso mapea los números de ID_ATC_subclase con su correspondencia a Nombre_ATC_subclase
display(df_final) # Print con mejor visualización

,ID_ATC_subclase,Nombre_ATC_subclase,ID_ATC,Nombre_med,#_RAFs
0,A01AB,Antiinfectives and antiseptics for local oral ...,A01AB03,Chlorhexidine,59
1,A01AB,Antiinfectives and antiseptics for local oral ...,A01AB04,Amphotericin b,348
2,A01AB,Antiinfectives and antiseptics for local oral ...,A01AB08,Neomycin,24
3,A01AB,Antiinfectives and antiseptics for local oral ...,A01AB09,Miconazole,70
4,A01AB,Antiinfectives and antiseptics for local oral ...,A01AB10,Natamycin,13
...,...,...,...,...,...
1566,V08AB,"Watersoluble, nephrotropic, low osmolar x-ray ...",V08AB09,Iodixanol,151
1567,V08CA,Paramagnetic contrast media,V08CA03,Gadodiamide,109
1568,V08CA,Paramagnetic contrast media,V08CA04,Gadoteridol,76
1569,V08CA,Paramagnetic contrast media,V08CA06,Gadoversetamide,203


7. Contar fármacos por categoría

In [55]:
final2 = df_final.value_counts("Nombre_ATC_subclase", sort = True) # Cuenta el número de filas asociadas a cada valor único de la columna indicada y los ordena en orden descendente
print(final2) # Imprime el resultado

Nombre_ATC_subclase
Antibiotics                                 36
Fluoroquinolones                            25
Corticosteroids                             25
Benzodiazepine derivatives                  25
Glucocorticoids                             24
                                            ..
Peripherally acting antiobesity products     1
Gonadotropin releasing hormone analogues     1
Penicillamine and similar agents             1
Hedgehog pathway inhibitors                  1
Zinc                                         1
Length: 403, dtype: int64


In [56]:
df_final2 = final2.rename_axis('Nombre_ATC_subclase').reset_index(name='#_meds') # Convierte la estructura de datos 'final2' en un dataframe
display(df_final2) # Print con mejor visualización

,Nombre_ATC_subclase,#_meds
0,Antibiotics,36
1,Fluoroquinolones,25
2,Corticosteroids,25
3,Benzodiazepine derivatives,25
4,Glucocorticoids,24
...,...,...
398,Peripherally acting antiobesity products,1
399,Gonadotropin releasing hormone analogues,1
400,Penicillamine and similar agents,1
401,Hedgehog pathway inhibitors,1


8. Graficar fármacos por categoría

In [57]:
alt.Chart(df_final2, title = alt.TitleParams('Número de medicamentos por subcategoría ATC en SIDER 4.1', anchor='middle')).mark_bar().encode(x = alt.X('Nombre_ATC_subclase', title = "Nombre de la subcategoría ATC"), y = alt.Y('#_meds', title = "Número de medicamentos por subcategoría ATC"), color = alt.value('blue'))

alt.Chart(...)